# Monitoring — Feature Drift & Stack Health

Batch deployments fail quietly: the job keeps producing lists while the
incoming population's feature distribution moves away from the training
baseline. The monitoring stage compares the scoring window against that
baseline — relative mean drift per feature — and flags anything that moved
enough to invalidate the model.

We also verify the infra contract: which containers this tutorial owns and
how to tear them down.


## Environment

In [ ]:
import os

# Kernel env on PATH so %%bash cells can call feast/aws/docker directly; the
# aws CLI and Feast binaries live in the workspace venv (/opt/mlops-venv).
os.environ["PATH"] = "/opt/mlops-venv/bin:" + os.environ["PATH"]
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

# This project's own moto (see docker-compose.yml) — NOT the lab's shared one.
os.environ["MOTO_URL"] = "http://telco-customer-churn-moto:5000"

## Stack health

Everything this tutorial runs is compose-managed: one moto container. The
SageMaker job/serving containers come and go per job — `docker container ls`
filtered by the `sagemaker.local` label shows any stragglers.


In [ ]:
%%bash
docker compose -f ../docker-compose.yml ps

In [ ]:
%%bash
docker container ls -a --filter label=sagemaker.local=true     --format 'table {{.Names}}	{{.Status}}	{{.Labels}}'

## Drift report

Baseline = the training-time feature distribution (the parquet the view
serves). Current = the population as it would look in a later window —
here simulated by re-sampling the snapshot with a known shift (tenure
drifts upward as the cohort ages, charges creep). Production would compare
two real snapshots; the mechanism is identical.


In [ ]:
import pandas as pd

baseline = pd.read_parquet("../data/processed/customer_features.parquet")
feature_cols = [
    c for c in baseline.columns if c not in ("customer_id", "event_timestamp")
]

# Simulated later window: same features, shifted means.
current = baseline.copy()
current["tenure_months"] = (current["tenure_months"] + 6).clip(0, 72)
current["monthly_charges"] = current["monthly_charges"] * 1.08
current["num_add_ons"] = (current["num_add_ons"] + 1).clip(0, 6)


def drift_row(name):
    b, c = baseline[name].mean(), current[name].mean()
    pct = (c - b) / b * 100 if b != 0 else (float("inf") if c != 0 else 0.0)
    if abs(pct) > 10:
        verdict = "drift"
    elif abs(pct) > 5:
        verdict = "mild"
    else:
        verdict = "ok"
    return name, round(b, 3), round(c, 3), round(pct, 1), verdict


report = pd.DataFrame(
    (drift_row(c) for c in feature_cols),
    columns=["feature", "baseline_mean", "current_mean", "drift_%", "verdict"],
).sort_values("drift_%", ascending=False, key=abs)
report

In [ ]:
import matplotlib.pyplot as plt

top = report.head(8).iloc[::-1]
plt.figure(figsize=(8, 4))
plt.barh(top["feature"], top["drift_%"], color="#cc241d")
plt.axvline(10, color="k", ls="--", lw=1)
plt.axvline(-10, color="k", ls="--", lw=1)
plt.xlabel("mean drift %")
plt.title("Feature drift — scoring window vs training baseline")
plt.show()

flagged = report[report["verdict"] == "drift"]
print(f"{len(flagged)} feature(s) beyond the 10% review line")
flagged

When `tenure_months`/`monthly_charges` drift past the review line the model
still runs — but its calibration degrades silently, which is the argument
for scheduled retraining rather than a one-shot deploy. A production gate
would page the owner; here the flagged table is the artifact.


## Teardown

Stop the project's compose services; the SageMaker job containers are
already gone (jobs exit), but any serving container left from an
interrupted run is removed by the label filter for completeness.


In [ ]:
%%bash
docker compose -f ../docker-compose.yml down

In [ ]:
%%bash
docker container ls --filter label=sagemaker.local.role=serve -q |     xargs -r docker rm -f

In [ ]:
%%bash
docker container ls -a --filter label=sagemaker.local=true     --format 'table {{.Names}}	{{.Status}}'